# Request analysis (task 4a) on Kaggle

RealEdit's simplification and taxonomy prompts in one call per request, with an open VLM (Qwen2.5-VL).

Before running:
- input: the private dataset `editjudge-data`;
- settings: GPU T4 x2, internet on;
- the code is cloned from GitHub (`BRANCH`), so push your changes first;
- choose `STEP` below and get the coordinator's approval for the run.

Steps (measured on Kaggle: qwen2.5-vl-3b takes about 2.3 s per request on one T4):

| STEP | what | time |
|---|---|---|
| `smoke` | 3B model, the fixed 20 requests | 2 min + model download |
| `compare` | 3B and 7B models on the fixed 100 requests, then their agreement | about 20 min |
| `full-3b` | 3B model on all 17,755 requests, one shard per GPU | about 6 h, one session |
| `full-7b` | 7B model on all requests, spread over both GPUs | to measure in `compare`; several sessions |

Every run stops by itself after `HOURS` (Kaggle ends a session at 12 h, and a session that is cut off keeps no
output). To go on in a new session: add this notebook's output as an input, set `PREVIOUS`, and run the same
`STEP` again; done requests are skipped.

In [ ]:
import os, shutil, subprocess, sys

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "curr"                            # or your task branch, e.g. "instructions/hai"
DATA = "/kaggle/input/editjudge-data"      # the private dataset (editjudge-data.zip, extracted by Kaggle)
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

os.environ["EJ_DATA_DIR"] = DATA           # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT          # all results are written here
os.makedirs(OUT, exist_ok=True)
os.chdir(CODE)

# Model weights go to /tmp: /kaggle/working is limited to 20 GB and is saved as the notebook's output
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:                                       # optional Kaggle secret HF_TOKEN (Add-ons > Secrets)
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:                          # public models need no token: hide the rate-limit notice
    os.environ["HF_HUB_VERBOSITY"] = "error"

PREVIOUS = None                            # e.g. "/kaggle/input/<this notebook's previous version>/out"
if PREVIOUS:
    shutil.copytree(f"{PREVIOUS}/instructions", f"{OUT}/instructions", dirs_exist_ok=True)

In [ ]:
!pip -q install -U "transformers>=4.56" accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m src.instructions list

In [ ]:
STEP = "smoke"          # smoke | compare | full-3b | full-7b
VERSION = 2             # prompt version (src/instructions/prompts/analyze_v<k>.txt)
HOURS = 11              # stop before Kaggle's 12 h session limit
MODELS = {"qwen2.5-vl-3b": "Qwen/Qwen2.5-VL-3B-Instruct", "qwen2.5-vl-7b": "Qwen/Qwen2.5-VL-7B-Instruct"}

def ej(*args):
    """python -m src.instructions <args>, printing its output."""
    subprocess.run([sys.executable, "-m", "src.instructions", *map(str, args)], check=True)

def run_on_gpus(client, gpus=2):
    """One process per GPU, each on its own shard; logs in OUT/logs/."""
    from huggingface_hub import snapshot_download
    snapshot_download(MODELS[client])      # download once, before the processes start
    os.makedirs(f"{OUT}/logs", exist_ok=True)
    procs = []
    for gpu in range(gpus):
        log = open(f"{OUT}/logs/{client}-v{VERSION}-gpu{gpu}.log", "a")
        cmd = [sys.executable, "-m", "src.instructions", "run", "--client", client, "--version", str(VERSION),
               "--shard", f"{gpu}/{gpus}", "--max-hours", str(HOURS)]
        procs.append(subprocess.Popen(cmd, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)),
                                      stdout=log, stderr=subprocess.STDOUT))
    for p in procs:
        p.wait()
    for gpu in range(gpus):
        print(open(f"{OUT}/logs/{client}-v{VERSION}-gpu{gpu}.log").read()[-600:])

In [ ]:
run = lambda client: f"{client}:analyze_v{VERSION}"

if STEP == "smoke":
    ej("run", "--client", "qwen2.5-vl-3b", "--version", VERSION, "--sample", 20)
    ej("stats", "--run", run("qwen2.5-vl-3b"))
elif STEP == "compare":
    for client in MODELS:
        ej("run", "--client", client, "--version", VERSION, "--sample", 100)
        ej("stats", "--run", run(client))
    ej("agree", "--run", run("qwen2.5-vl-3b"), "--run", run("qwen2.5-vl-7b"))
elif STEP == "full-3b":
    run_on_gpus("qwen2.5-vl-3b")           # the 3B model fits one T4: two shards side by side
    ej("stats", "--run", run("qwen2.5-vl-3b"))
elif STEP == "full-7b":
    ej("run", "--client", "qwen2.5-vl-7b", "--version", VERSION, "--max-hours", HOURS)   # one model over both T4
    ej("stats", "--run", run("qwen2.5-vl-7b"))

In [ ]:
!ls -lR /kaggle/working/out/instructions